In [1]:
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [24]:
# Image load ===> transform ==> dataset of all imgs

class ImageProcessor:
    def __init__(self, root_dir_path, transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        # list of paths for all images
        self.all_img_paths = [os.path.join(root_dir_path, img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, idx):
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [25]:
root_dir_path = "./img_align_celeba"

transformations = transforms.Compose([
    transforms.CenterCrop(178), # 178x178
    transforms.Resize(64),  # 64x64
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) #[-1, 1]
])

In [26]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded  {len(dataset)}  images")

loaded  202599  images


In [27]:
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# Generator Network

In [28]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [29]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, img_channels=3): # 3 is for RGB
        super(Generator, self).__init__()


        # fully connected (dense) layer
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256), # 100 ==> 256
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1024) ,
            nn.ReLU(),

            nn.Linear(1024, 64* 64 * img_channels) ,
            nn.Tanh(),
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64, 64)
        return img 
        # fake img => 64 x 64 x 3 x batch-size

# Discriminator Network

In [30]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=3): # 3 is for RGB
        super(Discriminator, self).__init__()


        # fully connected (dense) layer
        self.model = nn.Sequential(
            nn.Flatten(), # 3D tensor to 1D
            
            nn.Linear(img_channels* 64* 64, 1024),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(512, 256) ,
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(256, 1) ,
            nn.Sigmoid(),
        )

    def forward(self, img):
        return self.model(img)
       

In [31]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [32]:
import torch

# device
if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.device("cuda")

else:
    device = torch.device("cpu")

print(f"device is: {device}")

device is: cpu


In [33]:
generator = generator.to(device)
discriminator = discriminator.to(device)

# Training the GAN

In [37]:
def train(generator, discriminator, dataloader, epochs=10):

    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            # creates real imgs labels and fake imgs labels
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            # Train the Discriminator
            d_optimizer.zero_grad()

            fake_imgs = generator(torch.randn(batch_size, 100)).to(device)

            real_loss = GAN_loss(discriminator(real_imgs), real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()), real_labels)

            d_loss = (real_loss + fake_loss) / 2

            d_loss.backward()
            d_optimizer.step()

            # Train the Generator
            g_optimizer.zero_grad()
            g_loss = GAN_loss(discriminator(fake_imgs), real_labels)

            g_loss.backward()
            g_optimizer.step()

            if i%50 == 0:
                print(f"for epoch: {epoch+1}/{epochs}.... batch: {i+1}... G-loss: {g_loss}... D-loss: {d_loss}")

        # save generated imgs for each epoch
        save_generated_images(generator, epoch, device)
            

In [38]:
import matplotlib.pyplot as plt
import torchvision

def save_generated_images(generator, epoch, device, num_imgs=8):
    z = torch.randn(num_imgs, 100).to(device)
    generated_imgs = generator(z).detach().cpu()

    grid = torchvision.utils.make_grid(generated_imgs, nrow=4, normalize=True)

    plt.imshow(np.transepose(grid, (1,2,0)))
    plt.title(f"epoch {epoch+1}")
    plt.axis("off")
    plt.show()

In [39]:
train(generator, discriminator, dataloader, epochs=5)

for epoch: 1/5.... batch: 1... G-loss:0.6518790125846863... D-loss0.6726790070533752
for epoch: 1/5.... batch: 51... G-loss:7.705841198912822e-06... D-loss0.0006196329486556351
for epoch: 1/5.... batch: 101... G-loss:2.0777924873982556e-06... D-loss3.3011681807693094e-05
for epoch: 1/5.... batch: 151... G-loss:6.752102308382746e-07... D-loss0.00011975860979873687
for epoch: 1/5.... batch: 201... G-loss:2.775342409222503e-07... D-loss5.8506907407718245e-06
for epoch: 1/5.... batch: 251... G-loss:2.1234168912087625e-07... D-loss9.265141670766752e-06
for epoch: 1/5.... batch: 301... G-loss:7.729979500936679e-08... D-loss7.352895750045718e-07
for epoch: 1/5.... batch: 351... G-loss:6.426127896475009e-08... D-loss9.378893082612194e-06
for epoch: 1/5.... batch: 401... G-loss:5.494803900774059e-08... D-loss7.068053491821047e-06
for epoch: 1/5.... batch: 451... G-loss:2.51457130673316e-08... D-loss3.0522019187628757e-06
for epoch: 1/5.... batch: 501... G-loss:1.3038517820973539e-08... D-loss9.

AttributeError: module 'numpy' has no attribute 'transepose'